# Run Bayesian optimization for latent displacement

Optimize the paired validation ROC-AUC advantage of latent displacement over direct displacement on Default Credit at 5:1 imbalance. Results and optimizer state are persisted after every completed trial. Interrupt safely with Ctrl-C or the kernel stop button, then rerun the optimization cell to resume.

In [ ]:
PROFILE = "full"
DATASET_KEY = "default_credit"
IMBALANCE_RATIOS = (5.0,)
TRAINING_SIZE = 100
SEEDS = (0, 1)

CAPACITY_BOUNDS = (0.65, 1.0)
NEIGHBOR_BOUNDS = (10, 15)
LAMBDA_LOW_BOUNDS = (0.08, 0.30)
LAMBDA_WIDTH_BOUNDS = (0.40, 0.90)
MIMIC_MODE = "factorised"

N_TRIALS = 24
N_INITIAL_POINTS = 6
ACQUISITION_POOL_SIZE = 2048
OPTIMIZER_RANDOM_STATE = 0
VALIDATION_SIZE = 0.25
SHOW_PROGRESS = True
RESUME = True
RESTART_OPTIMIZATION = False
CONFIG_PATH = None
ARTIFACT_DIR = None

In [ ]:
# Explicit module reloads are used below so persistence code is never stale.

In [ ]:
from dataclasses import replace
import importlib
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from streamlined.config import load_config
from streamlined import latent_displacement_tuning as tuning
tuning = importlib.reload(tuning)

In [ ]:
config_path = Path(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
artifact_dir = ARTIFACT_DIR or str(EXPERIMENT_ROOT / "artifacts")
config = load_config(config_path, artifact_dir=artifact_dir)
config = replace(
    config,
    profile=replace(
        config.profile,
        datasets=(DATASET_KEY,),
        imbalance_ratios=IMBALANCE_RATIOS,
        training_sizes=(TRAINING_SIZE,),
        seeds=SEEDS,
    ),
)
space = tuning.BayesianOptimizationSpace(
    capacity_bounds=CAPACITY_BOUNDS,
    neighbor_bounds=NEIGHBOR_BOUNDS,
    lambda_low_bounds=LAMBDA_LOW_BOUNDS,
    lambda_width_bounds=LAMBDA_WIDTH_BOUNDS,
    mimic_mode=MIMIC_MODE,
)
print(f"Dataset: {DATASET_KEY}; ratio: {IMBALANCE_RATIOS}; training size: {TRAINING_SIZE}")
print(f"Seeds: {SEEDS}; trials: {N_TRIALS}; restart: {RESTART_OPTIMIZATION}")
print(f"Checkpoint: {tuning.default_optimization_checkpoint_path(config.artifact_root)}")

## Run or resume

Keep `RESTART_OPTIMIZATION = False` to resume. Set it to `True` for one execution only when intentionally discarding the current checkpoint and persisted results. Visualization is handled by `06_visualise_latent_displacement_optimization.ipynb`.

In [ ]:
tuning.run_bayesian_latent_displacement_optimization(
    config,
    dataset_key=DATASET_KEY,
    space=space,
    n_trials=N_TRIALS,
    n_initial_points=N_INITIAL_POINTS,
    acquisition_pool_size=ACQUISITION_POOL_SIZE,
    validation_size=VALIDATION_SIZE,
    random_state=OPTIMIZER_RANDOM_STATE,
    show_progress=SHOW_PROGRESS,
    checkpoint_path=tuning.default_optimization_checkpoint_path(config.artifact_root),
    resume=RESUME,
    restart=RESTART_OPTIMIZATION,
)